# Project 02 — Hospital Operational Efficiency Diagnostic

**Dataset:** CMS Hospital Compare — [Download here](https://data.cms.gov/provider-data/topics/hospitals)

This notebook runs fully on synthetic data that mirrors the CMS schema. When you have the real data, replace the data loading cell with the actual CSV paths.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['figure.figsize'] = (14, 5)
matplotlib.rcParams['axes.facecolor'] = '#111'
matplotlib.rcParams['figure.facecolor'] = '#0a0a0a'
matplotlib.rcParams['text.color'] = '#f0ede8'
matplotlib.rcParams['axes.labelcolor'] = '#a09d98'
matplotlib.rcParams['xtick.color'] = '#5a5755'
matplotlib.rcParams['ytick.color'] = '#5a5755'
matplotlib.rcParams['axes.edgecolor'] = '#2a2a2a'
matplotlib.rcParams['grid.color'] = '#1e1e1e'
print('Libraries loaded.')

## Step 1 — Load Data (Synthetic CMS-Schema)

In [ ]:
np.random.seed(99)
n = 340  # 340 peer hospitals

# Simulate CMS-style metrics
hospitals = pd.DataFrame({
    'hospital_id': [f'H{i:04d}' for i in range(n)],
    'hospital_name': [f'Community Hospital {i}' for i in range(n)],
    'state': np.random.choice(['CA','TX','FL','NY','IL','PA'], n),
    'beds': np.random.randint(100, 500, n),
    'is_network': [i < 8 for i in range(n)],  # first 8 are our network

    # Key quality metrics
    'readmission_rate_pct': np.clip(np.random.normal(14.5, 3.2, n), 6, 26),
    'hcahps_score': np.clip(np.random.normal(72, 9, n), 45, 95),
    'mortality_rate_pct': np.clip(np.random.normal(12.0, 2.5, n), 5, 22),
    'safety_score': np.clip(np.random.normal(68, 11, n), 30, 95),

    # Operational inputs (potential root causes)
    'nurse_staff_ratio': np.clip(np.random.normal(4.2, 0.9, n), 2.0, 7.5),
    'discharge_protocol_pct': np.clip(np.random.normal(74, 14, n), 30, 98),
    'icu_utilization_pct': np.clip(np.random.normal(72, 12, n), 40, 98),
    'avg_length_of_stay': np.clip(np.random.normal(4.8, 1.2, n), 2.5, 9.0),
    'spend_per_patient': np.clip(np.random.normal(14500, 3200, n), 7000, 28000),
})

# Make readmission correlate with nurse ratio and discharge protocol (for regression to find)
hospitals['readmission_rate_pct'] = (
    22 
    - 1.4 * hospitals['nurse_staff_ratio'] 
    - 0.065 * hospitals['discharge_protocol_pct'] 
    + 0.04 * hospitals['icu_utilization_pct']
    + np.random.normal(0, 1.5, n)
).clip(6, 26)

# Make 3 network hospitals underperform
hospitals.loc[hospitals['is_network'] & (hospitals.index < 3), 'readmission_rate_pct'] += 4.5
hospitals.loc[hospitals['is_network'] & (hospitals.index < 3), 'nurse_staff_ratio'] -= 0.9
hospitals.loc[hospitals['is_network'] & (hospitals.index < 3), 'discharge_protocol_pct'] -= 18

print(f'Dataset: {len(hospitals)} hospitals')
print(f'Network hospitals: {hospitals["is_network"].sum()}')
print(f'\nNetwork avg readmission rate: {hospitals[hospitals["is_network"]]["readmission_rate_pct"].mean():.1f}%')
print(f'Peer avg readmission rate: {hospitals[~hospitals["is_network"]]["readmission_rate_pct"].mean():.1f}%')

## Step 2 — Peer Benchmarking & Percentile Rankings

In [ ]:
peer_group = hospitals[~hospitals['is_network']].copy()
network = hospitals[hospitals['is_network']].copy()

metrics = ['readmission_rate_pct','hcahps_score','mortality_rate_pct','safety_score']

# Compute percentile ranks for network hospitals against peer group
for m in metrics:
    peer_vals = peer_group[m].values
    network[f'{m}_pctile'] = network[m].apply(
        lambda x: (peer_vals < x).mean() * 100 if m == 'readmission_rate_pct' or m == 'mortality_rate_pct'
        else (peer_vals < x).mean() * 100
    )

# Composite performance index (weighted)
# Lower readmission + mortality = better; higher HCAHPS + safety = better
network['perf_index'] = (
    (100 - network['readmission_rate_pct_pctile']) * 0.40 +
    network['hcahps_score_pctile'] * 0.30 +
    (100 - network['mortality_rate_pct_pctile']) * 0.20 +
    network['safety_score_pctile'] * 0.10
)

print('Network Hospitals — Performance vs Peer Group:')
print(network[['hospital_id','readmission_rate_pct','readmission_rate_pct_pctile','hcahps_score','perf_index']].round(1).to_string(index=False))
print(f'\nUnderperformers (perf_index < 40): {(network["perf_index"] < 40).sum()} hospitals')

In [ ]:
# Performance vs Spend 2x2 matrix
peer_med_spend = peer_group['spend_per_patient'].median()
peer_med_readmit = peer_group['readmission_rate_pct'].median()

fig, ax = plt.subplots(figsize=(12, 8))

ax.scatter(peer_group['spend_per_patient'], peer_group['readmission_rate_pct'],
           alpha=0.3, color='#5a5755', s=30, label='Peer hospitals')

colors = ['#c8f060' if r < 40 else '#f07060' for r in network['perf_index']]
ax.scatter(network['spend_per_patient'], network['readmission_rate_pct'],
           color=colors, s=120, zorder=5, label='Network hospitals', edgecolors='#f0ede8', linewidths=0.5)

ax.axvline(peer_med_spend, color='#2a2a2a', linestyle='--', linewidth=1)
ax.axhline(peer_med_readmit, color='#2a2a2a', linestyle='--', linewidth=1)

ax.text(peer_med_spend * 1.02, peer_group['readmission_rate_pct'].max() * 0.97,
        'HIGH SPEND\nHIGH READMIT', color='#f07060', fontsize=9, alpha=0.8)
ax.text(peer_group['spend_per_patient'].min(), peer_med_readmit * 0.85,
        'LOW SPEND\nLOW READMIT', color='#c8f060', fontsize=9, alpha=0.8)

ax.set_xlabel('Spend per Patient ($)')
ax.set_ylabel('30-Day Readmission Rate (%)')
ax.set_title('Performance vs Spend Matrix — Network vs Peer Group', color='#f0ede8', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig('benchmarking_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 3 — Regression Root Cause Analysis

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

features = ['nurse_staff_ratio','discharge_protocol_pct','icu_utilization_pct','avg_length_of_stay']
target = 'readmission_rate_pct'

X = hospitals[features].values
y = hospitals[target].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

reg = LinearRegression()
reg.fit(X_scaled, y)

y_pred = reg.predict(X_scaled)
r2 = r2_score(y, y_pred)

print(f'Model R²: {r2:.3f} ({r2*100:.1f}% of readmission variance explained)')
print(f'\nStandardized Coefficients (impact per 1 SD change):')
coef_df = pd.DataFrame({'Feature': features, 'Coefficient': reg.coef_, 'Direction': ['↑ bad' if c > 0 else '↓ good' for c in reg.coef_]})
coef_df = coef_df.sort_values('Coefficient', key=abs, ascending=False)
print(coef_df.to_string(index=False))

# Plot coefficients
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#f07060' if c > 0 else '#c8f060' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, alpha=0.85)
ax.axvline(0, color='#5a5755', linewidth=1)
ax.set_title(f'Root Cause Regression — Drivers of Readmission Rate (R²={r2:.2f})', color='#f0ede8', fontsize=12)
ax.set_xlabel('Standardized Coefficient (impact per 1 SD)')
plt.tight_layout()
plt.savefig('regression_root_cause.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n=== PROJECT 02 COMPLETE ===')
print(f'R² = {r2:.2f} | Top driver: {coef_df.iloc[0]["Feature"]}')